# Тест Бекдел

In [1]:
import re
import requests

In [2]:
INPUT_DIALOGUES = "../lab03/Zabrodin_file_7.txt"
UDPIPE_URL      = "http://localhost:3000/process"
BOOK_TITLE      = '"Кайдашева сім\'я"'
SEPARATOR       = "*" * 40

In [3]:

UA_NAME = r"[А-ЩЬЮЯҐЄІЇ][а-щьюяґєії'\-]+"

SPEAKER_INTRO_PATTERN  = re.compile(rf"({UA_NAME})\s+[^:]*:\s*$", re.MULTILINE) # Мотря сказала: - (...)
SPEAKER_INSIDE_PATTERN = re.compile(rf"—\s+[^—]+[,!?]\s+—\s+\w+\s+({UA_NAME})") # - (...), - сказала Мотря.

Читаємо діалоги з минулого завдання

In [4]:
def read_dialogues(filepath: str) -> list[str]:
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    return [b.strip() for b in content.split(SEPARATOR) if b.strip()]

Аналізуємо діалоги в UDPipe

In [5]:
def analyze(text: str) -> list[dict]:
    try:
        response = requests.post(UDPIPE_URL, data={
            "data": text,
            "tokenizer": "",
            "tagger": "",
            "parser": "",
        })
        response.raise_for_status()
    except requests.exceptions.ConnectionError:
        raise

    words = []
    for line in response.json()["result"].splitlines():
        if not line or line.startswith("#"):
            continue

        fields = line.split("\t")
        if len(fields) < 10 or "-" in fields[0]:
            continue

        words.append({
            "form":  fields[1],
            "lemma": fields[2],
            "upos":  fields[3],
            "feats": fields[5],
        })

    return words

Шукаємо можливих учасників діалогів

In [6]:
def find_speaker_names(block_text: str) -> set[str]:
    names = set()

    for m in SPEAKER_INTRO_PATTERN.finditer(block_text):
        names.add(m.group(1))

    for m in SPEAKER_INSIDE_PATTERN.finditer(block_text):
        names.add(m.group(1))

    return names

Перевіряємо, що потенційні учасники діалогу є живими людьми та прив'язуємо їх стать за допомогою морфологічних ознак.

In [7]:
def gender_map(words: list[dict]) -> dict[str, str]:
    gender = {}

    for w in words:
        if w["upos"] != "PROPN" or "Animacy=Anim" not in w["feats"]:
            continue

        if "Gender=Fem" in w["feats"]:
            gender[w["form"]]  = "Fem"
            gender[w["lemma"]] = "Fem"
        elif "Gender=Masc" in w["feats"]:
            gender[w["form"]]  = "Masc"
            gender[w["lemma"]] = "Masc"

    return gender

Перевіряємо чи є згадки осіб чоловічої статі

In [8]:
def has_male_mention(words: list[dict]) -> bool:
    return any(
        w["upos"] in ("NOUN", "PROPN") and
        "Animacy=Anim" in w["feats"] and
        "Gender=Masc" in w["feats"]
        for w in words
    )

In [9]:
def bechdel_test(blocks: list[str]) -> None:
    passed_blocks = []

    for block_text in blocks:
        names = find_speaker_names(block_text)
        if len(names) < 2:
            continue

        words = analyze(block_text)
        gender = gender_map(words)

        female_speakers = {name for name in names if gender.get(name) == "Fem"}
        if len(female_speakers) < 2:
            continue

        if not has_male_mention(words):
            passed_blocks.append((block_text, female_speakers))

    print(f"Блоків що проходять тест: {len(passed_blocks)}")

    if passed_blocks:
        print(f"{BOOK_TITLE} проходить тест Бекдел\n")
        for block_text, speakers in passed_blocks:
            print(f"Спікерки: {', '.join(speakers)}")
            print(block_text)
            print(SEPARATOR)
    else:
        print(f"{BOOK_TITLE} не проходить тест Бекдел")

In [10]:
blocks = read_dialogues(INPUT_DIALOGUES)
print(f"Зчитано блоків: {len(blocks)}\n")
bechdel_test(blocks)

Зчитано блоків: 326

Блоків що проходять тест: 5
"Кайдашева сім'я" проходить тест Бекдел

Спікерки: Кайдашиха, Мотря
— Легеньку руку маєш! Легенько ставиш, невістко! — крикнула Кайдашиха на Мотрю. — Одні ночви маємо, а ти й ті розбий.
— Як розіб' ю, то купите другі, — одрубала Мотря.
****************************************
Спікерки: Кайдашиха, Мотря
— Мабуть, хочете мене в черниці постригти, — сказала Мотря й кинула хустку на стіл. Вона глянула на матерію, набрану на спідницю; матерія була убога, темненька, з червоними краплями. Мотря навіть не розгорнула її та й одійшла од стола.
— Я знала, що тобі не вгожу. Я не знаю, хто тобі й вгодить, — сказала Кайдашиха, розсердившись, — де ж пак! Зросла в такій розкоші.
****************************************
Спікерки: Кайдашиха, Мотря
— Навіщо ти, Мотре, ховаєш починки в свою скриню? — спитала Кайдашиха.
— На те, що треба; не буду ж їх їсти, — одрубала Мотря.
— А може, й поїси: хто тебе знає, — сказала Кайдашиха.
— Не бійтесь, не понесу в шин